In [1]:
import numpy as np
import pandas as pd
import sklearn

import itertools
import warnings
import random
import os
import time
from datetime import datetime
from itertools import combinations
import copy
import gc
import glob
import sys

from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from mlxtend.classifier import EnsembleVoteClassifier
from bayes_opt import BayesianOptimization
from sklearn.base import clone

warnings.filterwarnings("ignore")

print('scikit-learn version:', sklearn.__version__)
print('numpy version:', np.__version__)
print('pandas version:', pd.__version__)
print('Python version:', sys.version)


scikit-learn version: 1.7.2
numpy version: 2.0.0
pandas version: 2.2.2
Python version: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:12:24) [GCC 11.2.0]


In [2]:
import traceback
from pathlib import Path

def log_exception(exception, name, run, file_path="error_log.txt"):
    """Log an exception with full traceback to a text file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    error_traceback = traceback.format_exc()
    error_message = f"""
    ===== ERROR {timestamp} =====
    Type: {type(exception).__name__}
    Dataset: {name}
    Run: {str(run)}
    Message: {str(exception)}
    Traceback:
    {error_traceback}
    =============================
    """
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    with open(file_path, "a", encoding="utf-8") as file:
        file.write(error_message)
    print(f"Exception logged to '{file_path}'")


In [3]:
from series_parallel import *


In [4]:
def get_model_names(model_list):
    return [str(m).split('(')[0] for m in model_list]

def get_name_from_model(model):
    return str(model).split('(')[0]

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)


def create_seed_models(number_trained_models=6, random_state=123):
    seed_everything(random_state)
    KNN = KNeighborsClassifier()
    SVClassifier = SVC(random_state=random_state)
    RIDGE = RidgeClassifier(random_state=random_state)
    DT = DecisionTreeClassifier(random_state=random_state)
    MLP = MLPClassifier(random_state=random_state)
    logreg = LogisticRegression(random_state=random_state)

    model_list = [KNN, SVClassifier, RIDGE, DT, MLP, logreg]
    return model_list[:number_trained_models]


def perform_bayesian_optimization(base_model, random_state,
                                  train_models_X, train_models_Y,
                                  train_ensemble_X, train_ensemble_Y,
                                  metric,
                                  init_points=10, n_iter=50, verbose=True):
    """Optimize hyperparameters of a base model using Bayesian optimization.

    Trains on train_models, evaluates on train_ensemble (validation set).
    Returns the best fitted model.
    """

    def model_evaluation(model, train_models_X, train_models_Y,
                         train_ensemble_X, train_ensemble_Y, metric, **params):
        if isinstance(model, KNeighborsClassifier):
            model.set_params(
                n_neighbors=int(params["n_neighbors"]),
                p=int(params["p"])
            )
        elif isinstance(model, SVC):
            model.set_params(
                C=10 ** params["C"],
                gamma=10 ** params.get("gamma", -4),
                kernel="rbf"
            )
        elif isinstance(model, RidgeClassifier):
            model.set_params(alpha=10 ** params["alpha"])
        elif isinstance(model, DecisionTreeClassifier):
            model.set_params(
                max_depth=int(params["max_depth"]),
                min_samples_split=int(params["min_samples_split"])
            )
        elif isinstance(model, MLPClassifier):
            model.set_params(
                hidden_layer_sizes=(int(params["hidden_layer_sizes"]),),
                alpha=10 ** params["alpha"],
                solver="lbfgs",
                activation="logistic",
                max_iter=10000
            )
        elif isinstance(model, LogisticRegression):
            model.set_params(
                C=10 ** params["C"],
                max_iter=int(params.get("max_iter", 1000))
            )
        else:
            raise ValueError(f"Unrecognized model: {model}")

        model.fit(train_models_X, train_models_Y)
        y_pred = model.predict(train_ensemble_X)
        return metric(train_ensemble_Y, y_pred)

    search_space = {
        "KNeighborsClassifier": {
            "n_neighbors": (1, 50),
            "p": (1, 3)
        },
        "SVC": {
            "C": (-4, 4),
            "gamma": (-4, 4)
        },
        "RidgeClassifier": {
            "alpha": (-4, 4),
        },
        "DecisionTreeClassifier": {
            "max_depth": (2, 30),
            "min_samples_split": (2, 25)
        },
        "MLPClassifier": {
            "hidden_layer_sizes": (1, 100),
            "alpha": (-4, 4)
        },
        "LogisticRegression": {
            "C": (-4, 4),
            "max_iter": (1000, 1000)  # Fixed value
        },
    }

    model_name = get_name_from_model(base_model)
    param_bounds = search_space[model_name]

    optimizer = BayesianOptimization(
        f=lambda **params: model_evaluation(
            model=clone(base_model),
            train_models_X=train_models_X,
            train_models_Y=train_models_Y,
            train_ensemble_X=train_ensemble_X,
            train_ensemble_Y=train_ensemble_Y,
            metric=metric,
            **params
        ),
        pbounds=param_bounds,
        allow_duplicate_points=True,
        random_state=random_state,
        verbose=False
    )

    if verbose:
        print("Optimizing base model:", model_name)
    optimizer.maximize(init_points=init_points, n_iter=n_iter)

    best_params = optimizer.max["params"]
    best_model = clone(base_model)

    if isinstance(best_model, KNeighborsClassifier):
        best_model.set_params(
            n_neighbors=int(best_params["n_neighbors"]),
            p=int(best_params["p"])
        )
    elif isinstance(best_model, SVC):
        best_model.set_params(
            C=10 ** best_params["C"],
            gamma=10 ** best_params.get("gamma", -4),
            kernel="rbf"
        )
    elif isinstance(best_model, RidgeClassifier):
        best_model.set_params(alpha=10 ** best_params["alpha"])
    elif isinstance(best_model, DecisionTreeClassifier):
        best_model.set_params(
            max_depth=int(best_params["max_depth"]),
            min_samples_split=int(best_params["min_samples_split"])
        )
    elif isinstance(best_model, MLPClassifier):
        best_model.set_params(
            hidden_layer_sizes=(int(best_params["hidden_layer_sizes"]),),
            alpha=10 ** best_params["alpha"],
            solver="lbfgs",
            activation="logistic",
            max_iter=10000
        )
    elif isinstance(best_model, LogisticRegression):
        best_model.set_params(
            C=10 ** best_params["C"],
            max_iter=int(best_params.get("max_iter", 1000))
        )
    else:
        raise ValueError(f"Unrecognized model: {type(best_model)}")

    best_model.fit(train_models_X, train_models_Y)
    return best_model


In [5]:
class dataset():
    def __init__(self, name_ds, metric=None, random_state=123,
                 range_number_models_ensemble=range(3, 7)):
        self.name_file = name_ds

        self.train_models_X, self.train_models_Y, \
        self.train_ensemble_X, self.train_ensemble_Y, self.test_X, self.test_Y = [None] * 6

        # Base models: trained, optimized with BO, sorted by validation score (desc)
        self.base_models_optimized_sorted_by_val = None

        # Ensemble containers
        self.voting_ensemble_hard = None
        self.voting_ensemble_hard_topN = None
        self.stacking_ensemble = None
        self.stacking_ensemble_topN = None

        # Baselines
        self.random_forest = None
        self.xgboost = None

        # Bayesian optimization budget
        self.bayesian_init_points = 2 #10 
        self.bayesian_n_iter = 10 #50

        self.random_state = random_state
        self.range_number_models_ensemble = range_number_models_ensemble

        if metric is None:
            self.metric = accuracy_score
        else:
            self.metric = metric
        self.dict_results = dict(name_dataset=name_ds, random_state=random_state,
                                 metric=str(self.metric))

    # ------------------------------------------------------------------
    # Data loading
    # ------------------------------------------------------------------

    def load_dataset(self):
        """Load dataset and split 60% train / 20% val / 20% test (stratified)."""
        X = pd.read_csv(self.name_file)
        y = X['target_end']
        X.drop(columns='target_end', inplace=True)

        le = LabelEncoder()
        y = le.fit_transform(y)

        self.train_models_X, self.test_X, \
        self.train_models_Y, self.test_Y = train_test_split(
            X.values, y, test_size=0.2, stratify=y,
            random_state=self.random_state)

        self.train_models_X, self.train_ensemble_X, \
        self.train_models_Y, self.train_ensemble_Y = train_test_split(
            self.train_models_X, self.train_models_Y,
            test_size=0.25, stratify=self.train_models_Y,
            random_state=self.random_state)
        # 0.25 × 0.8 = 0.2

    # ------------------------------------------------------------------
    # Metrics
    # ------------------------------------------------------------------

    def compute_metrics(self, model_name, model, train=False, val=False, test=True):
        """Compute binary F1 and accuracy on the requested splits."""
        results = {}
        if model is None:
            results[model_name + "_test"] = 0.0
            results[model_name + "_test_acc"] = 0.0
            print(model_name, "is None")
        else:
            if test:
                preds = model.predict(self.test_X)
                results[model_name + "_test"] = f1_score(self.test_Y, preds)
                results[model_name + "_test_acc"] = accuracy_score(self.test_Y, preds)
                print(model_name + "_test_f1", round(results[model_name + "_test"], 4))
            if train:
                preds = model.predict(self.train_models_X)
                results[model_name + "_train"] = f1_score(self.train_models_Y, preds)
            if val:
                preds = model.predict(self.train_ensemble_X)
                results[model_name + "_val"] = f1_score(self.train_ensemble_Y, preds)
        self.dict_results.update(results)

    # ------------------------------------------------------------------
    # Base models
    # ------------------------------------------------------------------

    def train_base_models(self):
        base_models = {}
        seed_models = create_seed_models(random_state=self.random_state)
        results = {}
        model_names = get_model_names(seed_models)

        for base_model, model_name in zip(seed_models, model_names):
            best_model = perform_bayesian_optimization(
                base_model, self.random_state,
                self.train_models_X, self.train_models_Y,
                self.train_ensemble_X, self.train_ensemble_Y,
                self.metric,
                init_points=self.bayesian_init_points,
                n_iter=self.bayesian_n_iter)
            base_models[model_name] = best_model
            results[model_name + "_optimized_train"] = self.metric(
                self.train_models_Y, best_model.predict(self.train_models_X))
            results[model_name + "_optimized_val"] = self.metric(
                self.train_ensemble_Y, best_model.predict(self.train_ensemble_X))
            self.compute_metrics(model_name + "_optimized", best_model)

        # Sort by validation score (descending)
        wanted_keys = [mn + "_optimized_val" for mn in model_names]
        dict_scores_val = {k.removesuffix("_optimized_val"): results[k]
                           for k in wanted_keys if k in results}
        dict_scores_val = dict(sorted(dict_scores_val.items(),
                                       key=lambda item: item[1], reverse=True))

        best_models_names = list(dict_scores_val.keys())
        self.base_models_optimized_sorted_by_val = [base_models[k] for k in best_models_names]

        # Store as strings to avoid memory leaks
        results["params_base_models"] = [str(m) for m in self.base_models_optimized_sorted_by_val]
        self.dict_results.update(results)

    # ------------------------------------------------------------------
    # Voting ensemble (hard only)
    # ------------------------------------------------------------------

    def train_voting_ensemble(self, only_top_n=False):
        label_prefix = "topN_" if only_top_n else ""
        list_best_voting_models = []

        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                model_combinations = [
                    tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])
                ]
            else:
                model_combinations = list(
                    combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))

            best_metric = -1
            best_voting = None

            for combo in model_combinations:
                models = list(combo)
                voting_ensemble = EnsembleVoteClassifier(
                    clfs=models,
                    fit_base_estimators=False
                ).fit(self.train_models_X, self.train_models_Y)

                val_score = self.metric(
                    self.train_ensemble_Y,
                    voting_ensemble.predict(self.train_ensemble_X))
                if val_score > best_metric:
                    best_metric = val_score
                    best_voting = voting_ensemble  # New object each iteration

            list_best_voting_models.append(best_voting)
            time_elapsed = time.time() - tic
            self.dict_results.update({
                "time_" + label_prefix + "voting_" + str(number_models_ensemble): time_elapsed
            })

        for voting_model, n in zip(list_best_voting_models, self.range_number_models_ensemble):
            self.compute_metrics(
                label_prefix + "voting_" + str(n),
                voting_model, train=True, val=True, test=True)

        if only_top_n:
            self.voting_ensemble_hard_topN = list_best_voting_models
        else:
            self.voting_ensemble_hard = list_best_voting_models

    # ------------------------------------------------------------------
    # Series-Parallel ensemble
    # ------------------------------------------------------------------

    def train_series_parallel(self, only_top_n=False):
        label_prefix = "topN_" if only_top_n else ""
        list_best_sp_models = []

        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                model_combinations = [
                    tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])
                ]
            else:
                model_combinations = list(
                    combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))

            best_metric = -1
            best_sp = None

            for combo in model_combinations:
                models = list(combo)
                estimators = [(get_name_from_model(m), m) for m in models]

                sp_ensemble = SeriesParallelEnsemble(estimators).fit(
                    self.train_ensemble_X, self.train_ensemble_Y,
                    metric=self.metric, n_splits=1)

                val_score = self.metric(
                    self.train_ensemble_Y,
                    sp_ensemble.predict(self.train_ensemble_X))
                if val_score > best_metric:
                    best_metric = val_score
                    best_sp = sp_ensemble 
                    
            list_best_sp_models.append(best_sp)
            time_elapsed = time.time() - tic
            self.dict_results.update({
                "time_" + label_prefix + "sp_" + str(number_models_ensemble): time_elapsed,
                label_prefix + "sp_tree_" + str(number_models_ensemble): best_sp.best_tree_as_string()
            })

        for sp_model, n in zip(list_best_sp_models, self.range_number_models_ensemble):
            self.compute_metrics(
                label_prefix + "sp_" + str(n),
                sp_model, train=True, val=True, test=True)

    # ------------------------------------------------------------------
    # Stacking ensemble
    # ------------------------------------------------------------------

    def _evaluate_stacking(self, base_models, C_log):
        """Evaluate stacking with cross-validation on VAL, optimizing C (in log scale)."""
        C = 10 ** C_log
        stack_model = StackingClassifier(
            estimators=base_models,
            final_estimator=LogisticRegression(C=C, max_iter=10000,
                                               random_state=self.random_state),
            cv="prefit"
        )

        cv_scores = []
        skf = StratifiedKFold(n_splits=5)
        for train_index, test_index in skf.split(self.train_ensemble_X, self.train_ensemble_Y):
            stack_model.fit(self.train_ensemble_X[train_index],
                            self.train_ensemble_Y[train_index])
            preds = stack_model.predict(self.train_ensemble_X[test_index])
            cv_scores.append(self.metric(self.train_ensemble_Y[test_index], preds))

        return np.mean(cv_scores)

    def _optimize_stacking(self, models):
        """Optimize the meta-model C parameter using Bayesian optimization."""
        pbounds = {"C_log": (-4, 4)}

        optimizer = BayesianOptimization(
            f=lambda C_log: self._evaluate_stacking(models, C_log),
            pbounds=pbounds,
            random_state=self.random_state,
            verbose=False
        )

        optimizer.maximize(init_points=self.bayesian_init_points,
                           n_iter=self.bayesian_n_iter)

        best_C_log = optimizer.max["params"]["C_log"]
        best_C = 10 ** best_C_log  # Convert to normal scale
        return best_C, optimizer.max["target"]

    def train_stacking_ensemble(self, only_top_n=False):
        label_prefix = "topN_" if only_top_n else ""
        stacking_list = []

        for number_models_ensemble in self.range_number_models_ensemble:
            tic = time.time()
            if only_top_n:
                model_combinations = [
                    tuple(self.base_models_optimized_sorted_by_val[:number_models_ensemble])
                ]
            else:
                model_combinations = list(
                    combinations(self.base_models_optimized_sorted_by_val, number_models_ensemble))

            best_metric = -1
            best_stacking = None

            for combo in model_combinations:
                models = list(combo)
                estimators = [(get_name_from_model(m), m) for m in models]

                best_C, best_score = self._optimize_stacking(estimators)

                final_stacking_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(
                                                                                C=best_C,  # Already in normal scale — NOT 10**best_C
                                                                                max_iter=10000, random_state=self.random_state),
                                                        cv="prefit"
                )
                final_stacking_model.fit(self.train_ensemble_X, self.train_ensemble_Y)

                val_score = self.metric(
                    self.train_ensemble_Y,
                    final_stacking_model.predict(self.train_ensemble_X))
                if val_score > best_metric:
                    best_metric = val_score
                    best_stacking = final_stacking_model

            stacking_list.append(best_stacking)
            time_elapsed = time.time() - tic
            self.dict_results.update({
                "time_" + label_prefix + "stacking_" + str(number_models_ensemble): time_elapsed
            })

        for stacking_model, n in zip(stacking_list, self.range_number_models_ensemble):
            self.compute_metrics(
                label_prefix + "stacking_" + str(n),
                stacking_model, train=True, val=True, test=True)

        if only_top_n:
            self.stacking_ensemble_topN = stacking_list
        else:
            self.stacking_ensemble = stacking_list

    # ------------------------------------------------------------------
    # Random Forest baseline
    # ------------------------------------------------------------------

    def train_random_forest(self):
        def evaluate_rf(n_estimators, max_depth, min_samples_split):
            model = RandomForestClassifier(
                n_estimators=int(n_estimators),
                max_depth=int(max_depth),
                min_samples_split=int(min_samples_split),
                random_state=self.random_state,
                n_jobs=-1
            )
            model.fit(self.train_models_X, self.train_models_Y)
            predictions = model.predict(self.train_ensemble_X)
            return self.metric(self.train_ensemble_Y, predictions)

        tic = time.time()
        pbounds = {
            "n_estimators": (10, 1000),
            "max_depth": (2, 20),
            "min_samples_split": (2, 25)
        }

        optimizer = BayesianOptimization(
            f=evaluate_rf,
            pbounds=pbounds,
            random_state=self.random_state,
            verbose=False
        )
        optimizer.maximize(init_points=self.bayesian_init_points,
                           n_iter=self.bayesian_n_iter)

        best_params = optimizer.max["params"]
        best_params = {
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "min_samples_split": int(best_params["min_samples_split"])
        }

        self.random_forest = RandomForestClassifier(
            **best_params,
            random_state=self.random_state,
            n_jobs=-1
        )
        self.random_forest.fit(self.train_models_X, self.train_models_Y)

        time_elapsed = time.time() - tic
        self.dict_results.update({
            "time_random_forest": time_elapsed,
            "random_forest_params": str(self.random_forest)  # String, not object
        })
        self.compute_metrics("random_forest", self.random_forest)

    # ------------------------------------------------------------------
    # XGBoost baseline
    # ------------------------------------------------------------------

    def train_xgboost(self):
        def evaluate_xgb(n_estimators, max_depth, subsample, colsample_bytree):
            model = XGBClassifier(
                n_estimators=int(n_estimators),
                max_depth=int(max_depth),
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                use_label_encoder=False,
                random_state=self.random_state,
                n_jobs=-1
            )
            model.fit(self.train_models_X, self.train_models_Y)
            predictions = model.predict(self.train_ensemble_X)
            return self.metric(self.train_ensemble_Y, predictions)

        tic = time.time()
        pbounds = {
            "n_estimators": (10, 1000),
            "max_depth": (2, 12),
            "subsample": (0.1, 1.0),
            "colsample_bytree": (0.1, 1.0)
        }

        optimizer = BayesianOptimization(
            f=evaluate_xgb,
            pbounds=pbounds,
            random_state=self.random_state,
            verbose=False
        )
        optimizer.maximize(init_points=self.bayesian_init_points,
                           n_iter=self.bayesian_n_iter)

        best_params = optimizer.max["params"]
        best_params = {
            "n_estimators": int(best_params["n_estimators"]),
            "max_depth": int(best_params["max_depth"]),
            "subsample": best_params["subsample"],
            "colsample_bytree": best_params["colsample_bytree"]
        }

        self.xgboost = XGBClassifier(
            **best_params,
            use_label_encoder=False,
            random_state=self.random_state,
            n_jobs=-1
        ).fit(self.train_models_X, self.train_models_Y)

        time_elapsed = time.time() - tic
        self.dict_results.update({
            "time_xgboost": time_elapsed,
            "xgboost_params": str(self.xgboost)  # String, not object
        })
        self.compute_metrics("xgboost", self.xgboost)

    # ------------------------------------------------------------------
    # Cleanup
    # ------------------------------------------------------------------

    def cleanup(self):
        """Free large objects to prevent memory leaks across runs."""
        self.train_models_X = None
        self.train_models_Y = None
        self.train_ensemble_X = None
        self.train_ensemble_Y = None
        self.test_X = None
        self.test_Y = None
        self.base_models_optimized_sorted_by_val = None
        self.voting_ensemble_hard = None
        self.voting_ensemble_hard_topN = None
        self.stacking_ensemble = None
        self.stacking_ensemble_topN = None
        self.random_forest = None
        self.xgboost = None
        gc.collect()

    # ------------------------------------------------------------------
    # Run all
    # ------------------------------------------------------------------

    def run(self, verbose=0):
        print("------ RUNNING DATASET", self.name_file, "------")
        self.load_dataset()
        self.train_base_models()

        # Series-Parallel: exhaustive search over all combinations
        print("Training Series-Parallel (all combinations)")
        self.train_series_parallel(only_top_n=False)
        # Series-Parallel: top-N only
        print("Training Series-Parallel (top-N)")
        self.train_series_parallel(only_top_n=True)

        # Voting (hard): exhaustive search
        print("Training Voting ensembles (all combinations)")
        self.train_voting_ensemble(only_top_n=False)
        # Voting (hard): top-N only
        print("Training Voting ensembles (top-N)")
        self.train_voting_ensemble(only_top_n=True)

        # Stacking: exhaustive + top-N
        print("Training Stacking ensembles (all combinations)")
        self.train_stacking_ensemble(only_top_n=False)
        print("Training Stacking ensembles (top-N)")
        self.train_stacking_ensemble(only_top_n=True)

        # Baselines
        print("Training Random Forest")
        self.train_random_forest()
        print("Training XGBoost")
        self.train_xgboost()


In [6]:
warnings.filterwarnings("ignore")

results = []
now = datetime.now()
NAME_FILE_CSV = 'results_binary_' + now.strftime("%Y_%m_%d_%H_%M_%S") + ".csv"
NRUNS = 10
RANGE_NUMBER_MODELS_ENSEMBLE = range(3, 7)

list_datasets = sorted(glob.glob("datasets_binary/*.csv"))
print("Datasets found:", list_datasets)
print("Number of runs:", NRUNS)

for run in range(NRUNS):
    for i in list_datasets:
        try:
            d = dataset(i, metric=f1_score,
                        random_state=123 * (run + 1),
                        range_number_models_ensemble=RANGE_NUMBER_MODELS_ENSEMBLE)
            d.run(verbose=True)
            d.dict_results.update({"run": run})
            results.append(copy.deepcopy(d.dict_results))
            d.cleanup()
            del d
            gc.collect()
            results_df = pd.DataFrame(results)
            results_df.to_csv(NAME_FILE_CSV, index=False)
        except Exception as e:
            print("Failed dataset", d.name_file)
            print(e)
            log_exception(e, d.name_file, run, "exceptions_binary.txt")


Datasets found: ['datasets_binary/Bioresponse.csv', 'datasets_binary/Contaminant.csv', 'datasets_binary/ECG5000.csv', 'datasets_binary/FICO.csv', 'datasets_binary/Insurance.csv', 'datasets_binary/Intersectional.csv', 'datasets_binary/Long.csv', 'datasets_binary/MagicTelescope.csv', 'datasets_binary/PhishingWebsites.csv', 'datasets_binary/Satellite.csv', 'datasets_binary/SpeedDating.csv', 'datasets_binary/Titanic.csv', 'datasets_binary/UCI.csv', 'datasets_binary/ada.csv', 'datasets_binary/churn.csv', 'datasets_binary/clean2.csv', 'datasets_binary/coil2000.csv', 'datasets_binary/compas.csv', 'datasets_binary/credit.csv', 'datasets_binary/default.csv', 'datasets_binary/dis.csv', 'datasets_binary/eeg.csv', 'datasets_binary/heloc.csv', 'datasets_binary/jasmine.csv', 'datasets_binary/jm1.csv', 'datasets_binary/kc1.csv', 'datasets_binary/kdd.csv', 'datasets_binary/kr.csv', 'datasets_binary/law.csv', 'datasets_binary/madeline.csv', 'datasets_binary/mc1.csv', 'datasets_binary/mushroom.csv', 'da